# GramPulse Phase 5: Numerical Forecasting Benchmark

This notebook evaluates forecasting models (Seasonal Naive, Moving Average, CatBoost, LightGBM) on the GramPulse Synthetic Enterprise Panel.

**Objective**: Predict `closing_cash_balance` 6 months into the future.
**Evaluation Metrics**: 3M/6M WAPE (Weighted Absolute Percentage Error) and Stress Recall across 4 holdout sets (Temporal, Geographic, Enterprise, and Shock).

In [1]:
!pip install catboost lightgbm pandas datasets numpy scikit-learn matplotlib seaborn

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error

plt.style.use("ggplot")
sns.set_palette("mako")

## 1. Data Loading & Preparation

In a live Kaggle environment, you can point this to your Hugging Face dataset or the local parquet files.

In [7]:
# Load dataset
# Change this path to your uploaded Kaggle dataset or HF path
dataset_path = "/kaggle/input/datasets/swarajchouriwar/grampulse-synthetic-pretrain"

try:
    df = pd.read_parquet(dataset_path)
    print(f"Loaded {len(df)} records.")
except:
    print("Dataset path not found. Please update `dataset_path` with your Kaggle path.")
    # Creating dummy DataFrame structure for compilation
    df = pd.DataFrame(columns=["enterprise_id", "month", "sector", "district", "closing_cash_balance", "operating_inflow", "operating_outflow", "dpd", "is_train", "is_temporal_holdout", "is_geo_holdout", "is_ent_holdout", "is_shock_holdout"])


Loaded 180000 records.


### Creating the t+6 Targets
We need to shift the data to create standard forecasting targets.

In [12]:
def prepare_forecasting_data(df):
    if df.empty: return df
    
    df = df.sort_values(["enterprise_id", "month"])
    
    # Target: t+6 cash balance
    df["target_closing_cash_t6"] = df.groupby("enterprise_id")["closing_cash_balance"].shift(-6)
    
    # Drop rows where we do not have a 6-month forward target
    df = df.dropna(subset=["target_closing_cash_t6"])
    return df

df_proc = prepare_forecasting_data(df)

## 2. Model Training

In [13]:
if not df_proc.empty:
    train_mask = df_proc["is_train"]
    train_df = df_proc[train_mask]
    
    features = ["operating_inflow", "operating_outflow", "closing_cash_balance"]
    X_train = train_df[features]
    y_train = train_df["target_closing_cash_t6"]
    
    print("Training CatBoost...")
    cb = CatBoostRegressor(iterations=200, verbose=0, random_seed=42)
    cb.fit(X_train, y_train)
    
    print("Training LightGBM...")
    lgbm = LGBMRegressor(n_estimators=100, random_state=42)
    lgbm.fit(X_train, y_train)


Training CatBoost...
Training LightGBM...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005131 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 88350, number of used features: 3
[LightGBM] [Info] Start training from score 27413.678044


## 3. Evaluation on Holdouts

We calculate **WAPE (Weighted Absolute Percentage Error)** and **Stress Recall** (identifying when a target drops below debt obligations).

In [14]:
def evaluate(model, test_df):
    if test_df.empty: return {}
    
    X_test = test_df[features]
    y_test = test_df["target_closing_cash_t6"]
    
    preds = model.predict(X_test)
    
    wape = np.sum(np.abs(y_test - preds)) / np.sum(np.abs(y_test))
    
    # Define stress as closing cash < 0 in t+6
    actual_stress = y_test < 0
    pred_stress = preds < 0
    
    true_positives = np.sum(actual_stress & pred_stress)
    actual_positives = np.sum(actual_stress)
    
    stress_recall = true_positives / actual_positives if actual_positives > 0 else 0
    
    return {"6M_WAPE": wape, "Stress_Recall": stress_recall}

if not df_proc.empty:
    holdouts = {
        "Temporal": df_proc[df_proc["is_temporal_holdout"]],
        "Enterprise OOD": df_proc[df_proc["is_ent_holdout"]],
        "Geographic OOD": df_proc[df_proc["is_geo_holdout"]],
        "Shock OOD": df_proc[df_proc["is_shock_holdout"]],
    }
    
    results = []
    for name, split_df in holdouts.items():
        cb_metrics = evaluate(cb, split_df)
        lgbm_metrics = evaluate(lgbm, split_df)
        
        results.append({
            "Holdout": name,
            "CatBoost 6M WAPE": cb_metrics.get("6M_WAPE", None),
            "CatBoost Stress Recall": cb_metrics.get("Stress_Recall", None),
            "LGBM 6M WAPE": lgbm_metrics.get("6M_WAPE", None),
            "LGBM Stress Recall": lgbm_metrics.get("Stress_Recall", None)
        })
        
    res_df = pd.DataFrame(results)
    display(res_df)


,Holdout,CatBoost 6M WAPE,CatBoost Stress Recall,LGBM 6M WAPE,LGBM Stress Recall
0,Temporal,NaN,NaN,NaN,NaN
1,Enterprise OOD,0.034027,0.933697,0.033932,0.931401
2,Geographic OOD,0.041081,0.915619,0.041192,0.910605
3,Shock OOD,0.046988,0.987554,0.047677,0.985400
